<a href="https://colab.research.google.com/github/SohaibWaheed21/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlyRank-Internship/week1-flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata

DC_Token = userdata.get("DC_Token")

print("DC_Token loaded:", DC_Token is not None)

DC_Token loaded: True


In [3]:
from huggingface_hub import login

login(token=DC_Token, add_to_git_credential=False)

print("Hugging Face login successful.")

Hugging Face login successful.


In [4]:
!pip -q install duckdb huggingface_hub

In [5]:
import duckdb

con = duckdb.connect()

print("DuckDB ready.")

DuckDB ready.


In [6]:
from huggingface_hub import HfApi

api = HfApi(token=DC_Token)

info = api.dataset_info("FlyRank/internship-warehouse")

print("Dataset:", info.id)
print("Access confirmed.")

Dataset: FlyRank/internship-warehouse
Access confirmed.


In [7]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=DC_Token
)

for f in files[:30]:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

## 1. Unit of analysis + time window

*One row = one content item for one client on one reporting date. I will use March 2026 as the development window from the daily performance warehouse table.*

In [8]:
from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=DC_Token
)

print("Downloaded:", march_file)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Downloaded: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [9]:
grain_check = con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet('{march_file}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").fetchdf()

print("Duplicate grain combinations found:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain combinations found: 0


,report_date,client_hash_id,content_hash_id,row_count


## 2. Fields: feature / label / context / excluded

### Features
- gsc_impressions — historical search visibility available at the decision moment.
- gsc_clicks — historical search clicks available at the decision moment.
- gsc_avg_position — observed search position available at the decision moment.
- ga4_sessions — historical sessions available at the decision moment.
- sessions_organic — historical organic sessions available at the decision moment.

### Label / proxy
- A future content-performance or decline outcome, defined from a later time window. It will not be taken from the same reporting window as the features.

### Context
- client_hash_id — identifies the client and can be used for grouping or splitting, but is not a model feature.
- content_hash_id — identifies the content item, but is not a model feature.
- report_date — identifies the reporting date and time window.
- month — identifies the warehouse partition/month.

### Excluded
- gsc_sum_position — excluded because gsc_avg_position already represents the average position and using both would be redundant.
- client_has_gsc, client_has_ga4, gsc_data_available, and ga4_data_available — used to understand data availability rather than as content-performance features.
- Future-window measurements — excluded because they would leak information from after the decision moment.

In [10]:
import duckdb

con = duckdb.connect()

columns = con.execute(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{march_file}')
""").fetchdf()

display(columns)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [11]:
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "sessions_organic",
]

context_cols = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "month",
]

excluded_cols = [
    "gsc_sum_position",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available",
]

print("Features:", feature_cols)
print("Context:", context_cols)
print("Excluded:", excluded_cols)

Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'sessions_organic']
Context: ['client_hash_id', 'content_hash_id', 'report_date', 'month']
Excluded: ['gsc_sum_position', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [12]:
grain_check = con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet('{march_file}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").fetchdf()

print("Duplicate grain combinations:", len(grain_check))
display(grain_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain combinations: 0


,report_date,client_hash_id,content_hash_id,row_count


In [13]:
count_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet('{march_file}')
""").fetchdf()

display(count_check)

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [14]:
availability_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet('{march_file}')
""").fetchdf()

display(availability_check)

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The March 2026 slice has uneven data availability. GSC data is available for 3,611,061 of 9,841,378 rows, while GA4 data is available for only 413,966 rows. Therefore, analyses using GA4 fields will cover a much smaller subset.

The daily table also represents a panel over reporting dates, so history is not necessarily equally deep for every client or content item. I will check data-availability flags before using GSC or GA4 measurements.

I will not treat missing data as zero without checking what the availability flags mean. I will also avoid using future-window measurements as features when defining a later outcome.

In [15]:
print("March rows:", 9_841_378)
print("GSC available rows:", 3_611_061)
print("GA4 available rows:", 413_966)

March rows: 9841378
GSC available rows: 3611061
GA4 available rows: 413966


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.